<a href="https://colab.research.google.com/github/Cyberpunk-San/ML-practice/blob/XGBoost/XGBoost_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

XGBoost-Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=300, n_features=2, noise=10, random_state=42)
y = (y - y.mean()) / y.std() # Normalize target for stable gradients

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

class XGBoostFromScratch:
    def __init__(self, n_estimators=50, lr=0.1, max_depth=3, reg_lambda=1):
        self.n_estimators = n_estimators
        self.lr = lr
        self.max_depth = max_depth
        self.reg_lambda = reg_lambda # L2 Regularization
        self.trees = []
        self.base_pred = None

    def _compute_gradients(self, y, y_pred):
        # For MSE Loss: L = 1/2 * (y - y_pred)^2
        # Gradient (g) = -(y - y_pred)
        # Hessian (h) = 1
        gradients = -(y - y_pred)
        hessians = np.ones_like(y)
        return gradients, hessians

    def fit(self, X, y):
        # 1. Initial prediction
        self.base_pred = np.mean(y)
        y_pred = np.full(y.shape, self.base_pred)

        for i in range(self.n_estimators):
            # 2. Compute Gradients and Hessians
            g, h = self._compute_gradients(y, y_pred)

            # 3. Fit tree to the negative gradient (residuals)
            # In real XGBoost, we'd use g and h to calculate split gain
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, -g)

            # 4. Update predictions with learning rate (Shrinkage)
            update = tree.predict(X)
            y_pred += self.lr * update
            self.trees.append(tree)

    def predict(self, X):
        y_pred = np.full(X.shape[0], self.base_pred)
        for tree in self.trees:
            y_pred += self.lr * tree.predict(X)
        return y_pred

In [ ]:
model = XGBoostFromScratch(n_estimators=100, lr=0.1)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mse = np.mean((y_test - preds)**2)

print(f"XGBoost Scratch MSE: {mse:.4f}")

Implementation

In [ ]:
plt.figure(figsize=(15, 5))

for i, count in enumerate([1, 10, 100]):
    plt.subplot(1, 3, i+1)
    temp_model = XGBoostFromScratch(n_estimators=count, lr=0.1)
    temp_model.fit(X_train, y_train)
    res = y_test - temp_model.predict(X_test)

    plt.scatter(range(len(res)), res, alpha=0.6, color='purple')
    plt.axhline(0, color='black', linestyle='--')
    plt.title(f"Residuals after {count} Trees")
    plt.ylim(-2, 2)

plt.tight_layout()
plt.show()

2D Visualization

In [ ]:
import plotly.graph_objects as go

x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), 50)
xx, yy = np.meshgrid(x_range, y_range)
grid = np.c_[xx.ravel(), yy.ravel()]

zz = model.predict(grid).reshape(xx.shape)

fig = go.Figure(data=[
    go.Surface(x=x_range, y=y_range, z=zz, colorscale='Viridis', opacity=0.8),
    go.Scatter3d(x=X_test[:, 0], y=X_test[:, 1], z=y_test, mode='markers',
                 marker=dict(size=3, color='red'))
])

fig.update_layout(title="XGBoost Scratch: 3D Prediction Surface",
                  scene=dict(xaxis_title='Feature 1', yaxis_title='Feature 2', zaxis_title='Target'))
fig.show()

3D Visualization

*Implemented the core of Gradient Boosting. We moved away from the "bagging" logic of Random Forests (where trees are independent) to a sequential learning process. Each tree we added was trained specifically to minimize the gradient of the loss function—essentially asking, "What is the fastest way to reduce the remaining error?" The 3D surface highlights the "Extreme" nature of the model: it creates a highly detailed, non-linear map of the data by stacking many simple decision rules. For your Edge AI or drone sensors, this is the most efficient way to capture complex physical patterns without needing the massive overhead of a Neural Network.*